# 📤 Upload Your z/OS MIPS Dataset

This notebook section helps you upload your own z/OS performance data.

**Required Format:** CSV file with the following columns:
- `application` - Application name/ID
- `timestamp` - Date/time (YYYY-MM-DD or YYYY-MM-DD HH:MM:SS)
- `M24H` - MIPS 24 hours
- `MDIU` - MIPS Diurne (daytime)
- `MPTE` - MIPS Pointe (peak period)
- `TXDIU` - Transaction rate DIU (x1000)
- `EFF` - Efficiency metric (0-1 or 0-100)
- `TVDIU` - Time value DIU
- `MIPS_consumption` - Target variable (actual MIPS consumption)

## 📋 Step 1: Download CSV Template (Optional)

Download a template to see the exact format required.

In [ ]:
import pandas as pd
from google.colab import files

# Create template CSV
template_data = {
    'application': ['APP_001', 'APP_002', 'APP_003'],
    'timestamp': ['2024-01-01', '2024-01-01', '2024-01-01'],
    'M24H': [1000.0, 1500.0, 900.0],
    'MDIU': [800.0, 1200.0, 750.0],
    'MPTE': [1200.0, 1800.0, 1100.0],
    'TXDIU': [50.0, 75.0, 45.0],
    'EFF': [0.95, 0.90, 0.92],
    'TVDIU': [100.0, 150.0, 90.0],
    'MIPS_consumption': [950.0, 1350.0, 850.0]
}

template_df = pd.DataFrame(template_data)
template_df.to_csv('mips_data_template.csv', index=False)

print("✅ Template created: mips_data_template.csv\n")
print("Preview:")
display(template_df)

print("\n📥 Downloading template...")
files.download('mips_data_template.csv')
print("\n💡 Use this template to format your data!")

## 📤 Step 2: Upload Your CSV File

Click "Choose Files" button below to upload your z/OS MIPS data.

In [ ]:
from google.colab import files
import pandas as pd
import os

print("="*70)
print("📤 UPLOAD YOUR Z/OS MIPS DATA")
print("="*70)
print("\n📋 Required columns:")
print("   - application, timestamp, M24H, MDIU, MPTE,")
print("   - TXDIU, EFF, TVDIU, MIPS_consumption")
print("\n🔍 File should be: CSV format, UTF-8 encoding")
print("📊 Recommended: 1000+ records for good results")
print("\n" + "="*70)

# Upload file
uploaded = files.upload()

if uploaded:
    # Get uploaded filename
    uploaded_filename = list(uploaded.keys())[0]
    
    print(f"\n✅ File received: {uploaded_filename}")
    print(f"   Size: {len(uploaded[uploaded_filename]) / 1024:.2f} KB")
    
    # Set as DATA_PATH for training
    DATA_PATH = uploaded_filename
    
    print(f"\n✅ DATA_PATH set to: {DATA_PATH}")
else:
    print("\n⚠️  No file uploaded. Using sample data instead.")
    DATA_PATH = None

## ✅ Step 3: Validate Uploaded Data

Let's check if your data has the correct format and structure.

In [ ]:
if DATA_PATH:
    print("="*70)
    print("🔍 VALIDATING YOUR DATA")
    print("="*70)
    
    try:
        # Load data
        data = pd.read_csv(DATA_PATH)
        
        print(f"\n✅ File loaded successfully!")
        print(f"   Records: {len(data):,}")
        print(f"   Columns: {len(data.columns)}")
        
        # Check required columns
        required_cols = ['application', 'M24H', 'MDIU', 'MPTE', 'TXDIU', 'EFF', 'TVDIU', 'MIPS_consumption']
        optional_cols = ['timestamp']
        
        missing_required = [col for col in required_cols if col not in data.columns]
        missing_optional = [col for col in optional_cols if col not in data.columns]
        
        print("\n📋 Column Check:")
        if not missing_required:
            print("   ✅ All required columns present")
        else:
            print(f"   ❌ Missing required columns: {missing_required}")
            print("\n   Available columns:", list(data.columns))
        
        if missing_optional:
            print(f"   ⚠️  Optional column missing: {missing_optional}")
            print("      (Will create default timestamp)")
        
        # Check data types
        print("\n📊 Data Types:")
        numeric_cols = ['M24H', 'MDIU', 'MPTE', 'TXDIU', 'EFF', 'TVDIU', 'MIPS_consumption']
        for col in numeric_cols:
            if col in data.columns:
                if pd.api.types.is_numeric_dtype(data[col]):
                    print(f"   ✅ {col}: {data[col].dtype}")
                else:
                    print(f"   ⚠️  {col}: {data[col].dtype} (should be numeric)")
        
        # Check for missing values
        missing_values = data.isnull().sum()
        if missing_values.sum() > 0:
            print("\n⚠️  Missing Values Detected:")
            for col, count in missing_values[missing_values > 0].items():
                pct = (count / len(data)) * 100
                print(f"   {col}: {count} ({pct:.1f}%)")
            print("\n   💡 Tip: Missing values will be imputed during preprocessing")
        else:
            print("\n✅ No missing values detected")
        
        # Display statistics
        print("\n📈 Data Preview:")
        display(data.head())
        
        print("\n📊 Basic Statistics:")
        display(data.describe())
        
        # Check data quality
        print("\n🎯 Data Quality Check:")
        
        # Number of applications
        if 'application' in data.columns:
            n_apps = data['application'].nunique()
            print(f"   Applications: {n_apps}")
        
        # Date range
        if 'timestamp' in data.columns:
            try:
                data['timestamp'] = pd.to_datetime(data['timestamp'])
                date_range = (data['timestamp'].max() - data['timestamp'].min()).days
                print(f"   Date range: {data['timestamp'].min().date()} to {data['timestamp'].max().date()}")
                print(f"   Duration: {date_range} days ({date_range/365:.1f} years)")
            except:
                print("   ⚠️  Could not parse timestamp column")
        
        # Target variable range
        if 'MIPS_consumption' in data.columns:
            print(f"   MIPS range: {data['MIPS_consumption'].min():.0f} - {data['MIPS_consumption'].max():.0f}")
            print(f"   MIPS mean: {data['MIPS_consumption'].mean():.2f}")
        
        # Recommendations
        print("\n💡 Recommendations:")
        if len(data) < 500:
            print("   ⚠️  Small dataset (<500 records). Consider collecting more data.")
        elif len(data) < 1000:
            print("   ⚠️  Moderate dataset (<1000 records). More data recommended.")
        else:
            print("   ✅ Good dataset size (1000+ records)")
        
        if not missing_required:
            print("   ✅ Data format is correct - ready for training!")
        else:
            print("   ❌ Please fix missing columns before training")
        
        print("\n" + "="*70)
        
    except Exception as e:
        print(f"\n❌ Error loading file: {str(e)}")
        print("\n💡 Troubleshooting tips:")
        print("   - Check file encoding (should be UTF-8)")
        print("   - Verify CSV format (comma-separated)")
        print("   - Ensure no special characters in column names")
        print("   - Download and fill the template above")
        DATA_PATH = None

else:
    print("\n⚠️  No file uploaded. Use the cell above to upload your data.")

## 🔧 Step 4: Fix Common Issues (If Needed)

Run this cell to automatically fix common data issues.

In [ ]:
if DATA_PATH:
    print("🔧 Attempting to fix common data issues...\n")
    
    data = pd.read_csv(DATA_PATH)
    original_rows = len(data)
    
    # 1. Add timestamp if missing
    if 'timestamp' not in data.columns:
        print("📅 Creating default timestamp column...")
        data['timestamp'] = pd.date_range(start='2022-01-01', periods=len(data), freq='D')
        print("   ✅ Timestamp column added")
    
    # 2. Convert efficiency to 0-1 scale if in 0-100
    if 'EFF' in data.columns:
        if data['EFF'].max() > 1.0:
            print("\n📊 Converting EFF from 0-100 scale to 0-1...")
            data['EFF'] = data['EFF'] / 100
            print(f"   ✅ EFF converted (new range: {data['EFF'].min():.2f} - {data['EFF'].max():.2f})")
    
    # 3. Remove rows with missing target
    if 'MIPS_consumption' in data.columns:
        missing_target = data['MIPS_consumption'].isnull().sum()
        if missing_target > 0:
            print(f"\n🗑️  Removing {missing_target} rows with missing MIPS_consumption...")
            data = data.dropna(subset=['MIPS_consumption'])
            print("   ✅ Rows removed")
    
    # 4. Remove duplicate rows
    duplicates = data.duplicated().sum()
    if duplicates > 0:
        print(f"\n🗑️  Removing {duplicates} duplicate rows...")
        data = data.drop_duplicates()
        print("   ✅ Duplicates removed")
    
    # 5. Save cleaned data
    cleaned_path = 'cleaned_' + DATA_PATH
    data.to_csv(cleaned_path, index=False)
    
    print(f"\n✅ Data cleaned and saved to: {cleaned_path}")
    print(f"   Original rows: {original_rows}")
    print(f"   Final rows: {len(data)}")
    print(f"   Rows removed: {original_rows - len(data)}")
    
    # Update DATA_PATH to use cleaned file
    DATA_PATH = cleaned_path
    print(f"\n✅ DATA_PATH updated to: {DATA_PATH}")
    print("\n🚀 Your data is ready for training!")

else:
    print("⚠️  No file uploaded. Upload your data first.")

## ✅ Data Ready!

Your data is now validated and ready for training.

**Next steps:**
1. Return to the main notebook
2. The variable `DATA_PATH` is set to your uploaded file
3. Continue with Step 4: Configure Training Pipeline

**Or run training immediately:**

In [ ]:
if DATA_PATH:
    print("="*70)
    print("🎯 READY TO TRAIN!")
    print("="*70)
    print(f"\nData file: {DATA_PATH}")
    print(f"Records: {len(pd.read_csv(DATA_PATH)):,}")
    print("\n💡 Options:")
    print("   1. Go back to main notebook and continue from Step 4")
    print("   2. Or copy DATA_PATH variable and use it in training")
    print("\n📋 DATA_PATH for copy-paste:")
    print(f"   DATA_PATH = '{DATA_PATH}'")
else:
    print("⚠️  Please upload your data using the cells above")